# Routing & cost attribution across providers

Aura fronts many providers behind one endpoint. Each response reports usage and cost
(`usage.cost_usd`), and `metadata.aura.provider` tells you which provider actually served
the request — handy for verifying fallback/routing rules and feeding cost dashboards
(the admin UI and `/metrics` expose the same data).

In [ ]:
from aura import AuraClient

client = AuraClient()

In [ ]:
# Hit three providers through one gateway endpoint
models = ["gpt-5.4-mini", "claude-sonnet-4-6", "gemini-2.5-flash"]

for model in models:
    response = client.responses.create(
        model=model,
        input="Explain what an LLM gateway does in one sentence.",
    )
    provider = None
    if response.metadata and response.metadata.aura:
        provider = response.metadata.aura.provider
    u = response.usage
    tokens = f"{u.input_tokens} in / {u.output_tokens} out" if u else "n/a"
    cost = f"${u.cost_usd:.6f}" if (u and u.cost_usd is not None) else "n/a"
    print(f"{model:22} provider={provider or 'unknown':10} {tokens:18} cost={cost}")

In [ ]:
# The same numbers power the admin dashboard and /metrics:
#   curl -s localhost:8080/metrics | grep aura_cost
print("See the admin dashboard and /metrics for aggregated cost.")

Cost attribution is per-request; routing rules decide which provider handles which
model. If `provider` comes back `unknown`, your gateway build predates the
`metadata.aura` block — upgrade and it appears automatically.

That's the full evaluator tour: quickstart → streaming → tools → compression → validation
→ feedback → routing/costs.